# Stage A / NB 00 — Environment, path resolution, and pinned model registry

Protocol reference: `resubmission_study_protocol_and_ablation_plan.txt`, Section 9, Stage A.

Purpose
1. Record the exact software and hardware environment (referee 2c, reproducibility).
2. Resolve every dataset and image root on Biowulf once, and publish the result as
   `stage_a_paths.json` so that NB 01-04 and Stages B-D never hard-code a path again.
3. Pin every model to an immutable Hugging Face commit SHA. `main` is never used.
4. Attempt one load and one forward pass per model, and record the outcome.

Outputs
- `environment_manifest.json`
- `stage_a_paths.json`
- `model_registry.csv` / `model_registry.json`  (-> manuscript Table S3)
- `gpu_benchmark.json`

Gate
- All REQUIRED models resolve to a commit SHA and complete a forward pass.
- All required dataset files resolve.
- Optional models may fail to load; the failure reason is recorded, not swallowed.

## 1. Imports and environment capture

No installation commands. Run in the already-repaired Biowulf environment and restart the
kernel first if packages were changed earlier in the session.

In [ ]:
import csv
import gc
import hashlib
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import time
from collections import OrderedDict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch

PACKAGES = [
    "torch", "torchvision", "transformers", "huggingface-hub", "peft", "accelerate",
    "datasets", "safetensors", "pillow", "numpy", "pandas", "scipy", "scikit-learn",
    "matplotlib", "bitsandbytes", "timm",
]

package_versions = {}
print("Python packages")
for package in PACKAGES:
    try:
        package_versions[package] = version(package)
    except Exception as exc:
        package_versions[package] = f"unavailable ({exc})"
    print(f"  {package}: {package_versions[package]}")

print()
print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Host:", socket.gethostname())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Torch CUDA:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"  [{index}] {properties.name} "
              f"{properties.total_memory / 1024**3:.1f} GiB "
              f"sm_{properties.major}{properties.minor}")
    print("BF16 supported:", torch.cuda.is_bf16_supported())

## 2. Configuration

`PROJECT_ROOT` is the Biowulf data root used by every tested notebook. `STAGE_ROOT` is the
new output tree for this resubmission so that Stage A never overwrites the earlier results
in `medgemma15_4b_multitask_lora_rank32_cv/` or `medgemma15_4b_cxr_bbox_lora_rank32/`.

`DATASET_SEARCH_DIRS` is a candidate list rather than a single path: the three source files
are edited locally and copied to the cluster, and the copy location has historically moved.
The first directory that contains a given file wins, and the resolved absolute path is
written to `stage_a_paths.json`.

In [ ]:
SEED = 42

PROJECT_ROOT = Path("/data/liangz2/openi/midrc")
STAGE_ROOT = PROJECT_ROOT / "tetci_resubmit"
STAGE_A_DIR = STAGE_ROOT / "stage_A"
NB00_DIR = STAGE_A_DIR / "nb00_environment"

# Legacy artifacts produced by the four tested notebooks. Read-only in Stage A.
LEGACY_MULTITASK_CV_DIR = PROJECT_ROOT / "multi_task_CV"
LEGACY_BBOX_ADAPTER_DIR = PROJECT_ROOT / "medgemma15_4b_cxr_bbox_lora_rank32" / "final_adapter"
LEGACY_MULTITASK_ADAPTER_ROOT = PROJECT_ROOT / "medgemma15_4b_multitask_lora_rank32_cv"

# Candidate directories that may hold the three uploaded source files.
DATASET_SEARCH_DIRS = [
    STAGE_ROOT / "datasets",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT,
    Path.cwd(),
    Path.cwd().parent / "datasets",
    Path.cwd().parent.parent / "datasets",
    Path("/data/liangz2/openi/datasets"),
]

REQUIRED_DATASET_FILES = {
    "covid_midrc_dataset_csv": "covid_midrc_dataset.csv",
    "combined_cxr_harmony_train_jsonl": "combined_cxr_harmony_train.jsonl",
    "montgomery_cxr_images_csv": "montgomery_cxr_images.csv",
}

# The uploaded CSV/JSONL files store /vf/users paths; Biowulf compute nodes see /data.
IMAGE_PATH_REWRITES = OrderedDict([
    ("/vf/users/liangz2/openi", "/data/liangz2/openi"),
])

# Image roots that must exist for NB 01, NB 03, and NB 04.
IMAGE_ROOTS = {
    "midrc_xr_png": PROJECT_ROOT / "XR_png",
    "montgomery": PROJECT_ROOT / "Montgomery_image",
}

# One entry per model in the protocol. backend selects the loader in Section 5.
# required=True models gate this notebook; required=False models are recorded and reported.
MODEL_REGISTRY_SPEC = [
    {
        "role": "A1/A2 medical VLM and localizer base",
        "repo_id": "google/medgemma-1.5-4b-it",
        "backend": "image_text_to_text",
        "dtype": "bfloat16",
        "required": True,
        "gated": True,
        "note": "Base for the rank-32 anatomy/pathology bbox LoRA and the multitask LoRA.",
    },
    {
        "role": "A3 general-purpose VLM",
        "repo_id": "Qwen/Qwen3.5-4B",
        "backend": "multimodal_lm",
        "dtype": "bfloat16",
        "required": True,
        "gated": False,
        "note": "Hybrid full/linear attention text backbone; matched-scale control for MedGemma.",
    },
    {
        "role": "A4 reasoning VLM",
        "repo_id": "nvidia/NV-Reason-CXR-3B",
        "backend": "multimodal_auto",
        "dtype": "bfloat16",
        "required": True,
        "gated": False,
        "note": "Qwen2.5-VL-3B based chain-of-thought CXR model; zero-shot in E0c.",
    },
    {
        "role": "A5 frozen SSL encoder (sole frozen-encoder agent; carries RQ3)",
        "repo_id": "m42-health/CXformer-base",
        "backend": "vision_backbone",
        "dtype": "float32",
        "required": False,
        "gated": False,
        "note": (
            "DINOv2-derived ViT-B, 87M, SSL on 600K CXRs. May require trust_remote_code "
            "or timm on Biowulf. Marked optional only because its loader is the least "
            "standard; a load failure still blocks NB 05 and must be resolved, since this "
            "is now the only frozen-encoder arm."
        ),
    },
    # google/cxr-foundation was withdrawn (protocol D2a): its ELIXR-family encoder shares
    # Google's CXR representation lineage with the MedGemma-1.5 vision tower, so it would
    # have contributed a second Google embedding rather than an independent view, and it
    # was the study's only TensorFlow/Keras SavedModel dependency.
]

# Set False on a login node or when the cache is already populated and you only want
# the manifest refreshed.
RUN_MODEL_LOAD_CHECK = True
RUN_GPU_BENCHMARK = True

for directory in [STAGE_ROOT, STAGE_A_DIR, NB00_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Stage A root:", STAGE_A_DIR)
print("NB 00 output:", NB00_DIR)

## 3. Determinism

Seeds and deterministic kernels are set here and re-set identically at the top of every
later notebook. `CUBLAS_WORKSPACE_CONFIG` must be exported before the first CUDA call for
`use_deterministic_algorithms` to work, so it is set defensively and the resulting state is
recorded rather than assumed.

In [ ]:
import random

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", str(SEED))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

deterministic_algorithms = False
deterministic_error = None
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
    deterministic_algorithms = True
except Exception as exc:
    deterministic_error = str(exc)

determinism = {
    "seed": SEED,
    "pythonhashseed": os.environ.get("PYTHONHASHSEED"),
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    "cudnn_benchmark": torch.backends.cudnn.benchmark,
    "cudnn_deterministic": torch.backends.cudnn.deterministic,
    "use_deterministic_algorithms": deterministic_algorithms,
    "use_deterministic_algorithms_error": deterministic_error,
    "note": (
        "Generative evaluation uses do_sample=False everywhere except experiment family E6, "
        "which deliberately sweeps temperature and top_p."
    ),
}
print(json.dumps(determinism, indent=2))

## 4. Hugging Face authentication

MedGemma is gated: accept its licence on the Hugging Face website with this account
before running. The token is stored by the Hugging Face client and is not written into the
notebook.

In [ ]:
from huggingface_hub import HfApi, is_offline_mode, notebook_login, snapshot_download, whoami

HF_IDENTITY = None
HF_AUTH_ERROR = None
try:
    HF_IDENTITY = whoami()
except Exception:
    notebook_login()  # Paste your hf_... token into the widget.
    try:
        HF_IDENTITY = whoami()
    except Exception as exc:
        HF_AUTH_ERROR = str(exc)

print("Offline mode:", is_offline_mode())
if HF_IDENTITY is not None:
    print("Logged in as:", HF_IDENTITY.get("name"))
else:
    print("Not authenticated:", HF_AUTH_ERROR)
    print("Gated repositories (MedGemma) will fail to resolve.")

print("HF_HOME:", os.environ.get("HF_HOME", "<default>"))
print("HF_HUB_CACHE:", os.environ.get("HF_HUB_CACHE", "<default>"))

## 5. Resolve dataset files and image roots

The three uploaded source files are located, hashed, and recorded. Hashing matters: NB 01
and NB 02 assert that the file they read has the same SHA-256 recorded here, which prevents
a silently re-uploaded CSV from invalidating the folds halfway through Stage B.

In [ ]:
def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_dataset_file(filename, search_dirs):
    tried = []
    for directory in search_dirs:
        candidate = Path(directory) / filename
        tried.append(str(candidate))
        if candidate.is_file():
            return candidate.resolve(), tried
    return None, tried


dataset_paths = {}
dataset_records = []
missing_datasets = []

for key, filename in REQUIRED_DATASET_FILES.items():
    resolved, tried = resolve_dataset_file(filename, DATASET_SEARCH_DIRS)
    if resolved is None:
        missing_datasets.append((filename, tried))
        dataset_records.append({
            "key": key, "filename": filename, "path": None,
            "bytes": None, "sha256": None, "status": "MISSING",
        })
        continue
    dataset_paths[key] = str(resolved)
    dataset_records.append({
        "key": key,
        "filename": filename,
        "path": str(resolved),
        "bytes": resolved.stat().st_size,
        "sha256": sha256_file(resolved),
        "status": "OK",
    })

for record in dataset_records:
    if record["status"] == "OK":
        print(f"[OK]      {record['filename']}  {record['bytes']:,} bytes  "
              f"sha256={record['sha256'][:16]}...")
        print(f"          {record['path']}")
    else:
        print(f"[MISSING] {record['filename']}")

if missing_datasets:
    print()
    print("Searched directories:")
    for filename, tried in missing_datasets:
        print(f"  {filename}")
        for candidate in tried:
            print(f"    - {candidate}")
    print()
    print("Copy the missing files to one of the directories above, or extend "
          "DATASET_SEARCH_DIRS, then re-run this cell.")

In [ ]:
image_root_status = {}
for name, root in IMAGE_ROOTS.items():
    exists = root.is_dir()
    count = None
    if exists:
        # Cheap probe rather than a full recursive walk over ~2,600 study directories.
        count = sum(1 for _ in root.glob("*"))
    image_root_status[name] = {
        "path": str(root), "exists": exists, "top_level_entries": count,
    }
    flag = "OK" if exists else "MISSING"
    print(f"[{flag}] image root {name}: {root} (top-level entries: {count})")

legacy_status = {
    "multitask_cv_dir": {
        "path": str(LEGACY_MULTITASK_CV_DIR),
        "exists": LEGACY_MULTITASK_CV_DIR.is_dir(),
        "fold_files_present": sorted(
            path.name for path in LEGACY_MULTITASK_CV_DIR.glob("multitask_*_harmony.jsonl")
        ) if LEGACY_MULTITASK_CV_DIR.is_dir() else [],
    },
    "bbox_adapter_dir": {
        "path": str(LEGACY_BBOX_ADAPTER_DIR),
        "exists": LEGACY_BBOX_ADAPTER_DIR.joinpath("adapter_config.json").is_file(),
    },
    "multitask_adapter_root": {
        "path": str(LEGACY_MULTITASK_ADAPTER_ROOT),
        "folds_with_best_adapter": [
            fold for fold in range(5)
            if (LEGACY_MULTITASK_ADAPTER_ROOT / f"fold_{fold}" / "best_adapter"
                / "adapter_config.json").is_file()
        ],
    },
}
print()
print(json.dumps(legacy_status, indent=2))

## 6. GPU capability benchmark

Measured BF16 matmul throughput and peak memory. Stage B and Stage C quote wall-clock and
memory figures in manuscript Table 10; those numbers are only interpretable next to the
hardware they were produced on.

In [ ]:
gpu_benchmark = {"ran": False, "reason": None}

if RUN_GPU_BENCHMARK and torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    size = 4096
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    left = torch.randn(size, size, device="cuda", dtype=dtype)
    right = torch.randn(size, size, device="cuda", dtype=dtype)
    for _ in range(3):
        torch.matmul(left, right)
    torch.cuda.synchronize()
    start = time.perf_counter()
    iterations = 20
    for _ in range(iterations):
        torch.matmul(left, right)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    flops = 2.0 * size ** 3 * iterations
    properties = torch.cuda.get_device_properties(0)
    gpu_benchmark = {
        "ran": True,
        "reason": None,
        "device_name": properties.name,
        "total_memory_gib": round(properties.total_memory / 1024 ** 3, 2),
        "compute_capability": f"{properties.major}.{properties.minor}",
        "multi_processor_count": properties.multi_processor_count,
        "matmul_dtype": str(dtype),
        "matmul_size": size,
        "iterations": iterations,
        "seconds": round(elapsed, 4),
        "tflops": round(flops / elapsed / 1e12, 2),
        "peak_memory_gib": round(torch.cuda.max_memory_allocated() / 1024 ** 3, 3),
    }
    del left, right
    gc.collect()
    torch.cuda.empty_cache()
elif not torch.cuda.is_available():
    gpu_benchmark["reason"] = "CUDA not available on this node."
else:
    gpu_benchmark["reason"] = "RUN_GPU_BENCHMARK is False."

print(json.dumps(gpu_benchmark, indent=2))

disk = shutil.disk_usage(str(PROJECT_ROOT)) if PROJECT_ROOT.is_dir() else None
if disk is not None:
    print()
    print(f"Disk at {PROJECT_ROOT}: "
          f"{disk.free / 1024**3:.1f} GiB free of {disk.total / 1024**3:.1f} GiB")
    print("Localization views in NB 04 need roughly 3x the size of the source PNG tree.")

## 7. Pin every model to a commit SHA and attempt one forward pass

Two distinct operations, recorded separately so that a network failure is never mistaken
for a model incompatibility:

1. **Resolve** — `HfApi().model_info(repo_id)` returns the current commit SHA. That SHA,
   not `main`, is what every later notebook passes as `revision=`.
2. **Load check** — download at the pinned revision, build the processor and model, and run
   one forward pass on a synthetic 512x512 grey image.

A `required` model that fails either operation fails the gate in Section 9.

In [ ]:
from PIL import Image

PROBE_IMAGE = Image.new("RGB", (512, 512), (128, 128, 128))
PROBE_PROMPT = "Describe this chest radiograph in one sentence."

DTYPES = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}


def resolve_revision(repo_id):
    api = HfApi()
    info = api.model_info(repo_id)
    return {
        "sha": info.sha,
        "last_modified": str(getattr(info, "lastModified", None)),
        "gated": bool(getattr(info, "gated", False) or False),
        "private": bool(getattr(info, "private", False) or False),
        "n_siblings": len(getattr(info, "siblings", []) or []),
    }


def build_chat_prompt(processor, prompt_text):
    messages = [{
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": prompt_text}],
    }]
    return processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )


def load_and_probe(spec, revision):
    # Returns (status, detail_dict). Never raises; the caller decides what is fatal.
    repo_id = spec["repo_id"]
    backend = spec["backend"]
    torch_dtype = DTYPES[spec["dtype"]]
    detail = {"backend": backend, "loaded": False, "forward_ok": False}

    if backend == "download_only":
        try:
            local_dir = snapshot_download(repo_id=repo_id, revision=revision)
            detail["local_dir"] = local_dir
            detail["note"] = "Downloaded only; loaded by NB 05 with its native runtime."
            return "DOWNLOADED", detail
        except Exception as exc:
            detail["error"] = f"{type(exc).__name__}: {exc}"
            return "FAILED", detail

    model = None
    try:
        from transformers import AutoConfig, AutoModel, AutoProcessor
        common = {"revision": revision, "trust_remote_code": True}

        if backend == "vision_backbone":
            from transformers import AutoImageProcessor
            processor = AutoImageProcessor.from_pretrained(repo_id, **common)
            model = AutoModel.from_pretrained(repo_id, torch_dtype=torch_dtype, **common)
            model.eval()
            detail["loaded"] = True
            inputs = processor(images=PROBE_IMAGE, return_tensors="pt")
            with torch.inference_mode():
                outputs = model(**inputs)
            hidden = getattr(outputs, "last_hidden_state", None)
            if hidden is None:
                hidden = getattr(outputs, "pooler_output", None)
            detail["output_shape"] = list(hidden.shape) if hidden is not None else None
            detail["forward_ok"] = hidden is not None
        else:
            if backend == "image_text_to_text":
                from transformers import AutoModelForImageTextToText as ModelClass
            elif backend == "multimodal_lm":
                from transformers import AutoModelForMultimodalLM as ModelClass
            else:
                try:
                    from transformers import AutoModelForImageTextToText as ModelClass
                except Exception:
                    from transformers import AutoModelForVision2Seq as ModelClass

            processor = AutoProcessor.from_pretrained(repo_id, **common)
            if processor.tokenizer.pad_token_id is None:
                processor.tokenizer.pad_token = processor.tokenizer.eos_token
            model = ModelClass.from_pretrained(
                repo_id, torch_dtype=torch_dtype, device_map="auto",
                low_cpu_mem_usage=True, **common,
            )
            model.eval()
            detail["loaded"] = True
            detail["model_class"] = type(model).__name__

            prompt = build_chat_prompt(processor, PROBE_PROMPT)
            inputs = processor(text=prompt, images=PROBE_IMAGE, return_tensors="pt")
            device = next(
                (p.device for p in model.parameters() if p.device.type not in {"meta", "cpu"}),
                torch.device("cpu"),
            )
            moved = {
                key: (value.to(device=device, dtype=torch_dtype)
                      if value.is_floating_point() else value.to(device))
                for key, value in inputs.items()
            }
            with torch.inference_mode():
                output_ids = model.generate(
                    **moved, do_sample=False, max_new_tokens=8,
                    pad_token_id=processor.tokenizer.pad_token_id,
                )
            prompt_length = moved["input_ids"].shape[-1]
            detail["probe_completion"] = processor.decode(
                output_ids[0, prompt_length:], skip_special_tokens=True
            ).strip()
            detail["forward_ok"] = True

        total = sum(p.numel() for p in model.parameters())
        detail["total_parameters"] = int(total)
        detail["total_parameters_billions"] = round(total / 1e9, 3)
        if torch.cuda.is_available():
            detail["peak_memory_gib"] = round(
                torch.cuda.max_memory_allocated() / 1024 ** 3, 3
            )
        return "OK", detail

    except Exception as exc:
        detail["error"] = f"{type(exc).__name__}: {exc}"
        return "FAILED", detail
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

In [ ]:
model_registry = []

for spec in MODEL_REGISTRY_SPEC:
    repo_id = spec["repo_id"]
    print("=" * 78)
    print(repo_id, f"({spec['role']})")
    entry = {
        "repo_id": repo_id,
        "role": spec["role"],
        "backend": spec["backend"],
        "dtype": spec["dtype"],
        "required": spec["required"],
        "declared_gated": spec["gated"],
        "note": spec["note"],
        "revision": None,
        "resolve_status": None,
        "load_status": "SKIPPED",
        "detail": {},
    }

    try:
        info = resolve_revision(repo_id)
        entry["revision"] = info["sha"]
        entry["hub_last_modified"] = info["last_modified"]
        entry["hub_gated"] = info["gated"]
        entry["hub_files"] = info["n_siblings"]
        entry["resolve_status"] = "OK"
        print(f"  revision: {info['sha']}")
    except Exception as exc:
        entry["resolve_status"] = "FAILED"
        entry["detail"]["resolve_error"] = f"{type(exc).__name__}: {exc}"
        print(f"  resolve FAILED: {exc}")

    if entry["resolve_status"] == "OK" and RUN_MODEL_LOAD_CHECK:
        started = time.perf_counter()
        status, detail = load_and_probe(spec, entry["revision"])
        detail["load_seconds"] = round(time.perf_counter() - started, 1)
        entry["load_status"] = status
        entry["detail"].update(detail)
        print(f"  load: {status} ({detail['load_seconds']}s)")
        if detail.get("total_parameters_billions") is not None:
            print(f"  parameters: {detail['total_parameters_billions']}B")
        if detail.get("probe_completion") is not None:
            print(f"  probe completion: {detail['probe_completion']!r}")
        if detail.get("output_shape") is not None:
            print(f"  output shape: {detail['output_shape']}")
        if status == "FAILED":
            print(f"  error: {detail.get('error')}")

    model_registry.append(entry)

print("=" * 78)
print()
for entry in model_registry:
    marker = "REQUIRED" if entry["required"] else "optional"
    print(f"{entry['repo_id']:<34} {marker:<9} resolve={entry['resolve_status']:<7} "
          f"load={entry['load_status']}")

## 8. Write the manifests

`stage_a_paths.json` is the contract for NB 01-04 and for Stages B-D. Later notebooks read
it instead of redefining paths, so a cluster path change is a one-line fix in NB 00.

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(json_safe(payload), handle, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)
    return path


def git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=str(Path.cwd()),
            stderr=subprocess.DEVNULL, text=True,
        ).strip()
    except Exception:
        return None


stage_a_paths = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "project_root": str(PROJECT_ROOT),
    "stage_root": str(STAGE_ROOT),
    "stage_a_dir": str(STAGE_A_DIR),
    "nb_output_dirs": {
        "nb00_environment": str(NB00_DIR),
        "nb01_inventory": str(STAGE_A_DIR / "nb01_inventory"),
        "nb02_folds": str(STAGE_A_DIR / "nb02_folds"),
        "nb03_external": str(STAGE_A_DIR / "nb03_external"),
        "nb04_localization": str(STAGE_A_DIR / "nb04_localization"),
    },
    "datasets": dataset_paths,
    "dataset_hashes": {
        record["key"]: record["sha256"] for record in dataset_records
        if record["status"] == "OK"
    },
    "image_path_rewrites": dict(IMAGE_PATH_REWRITES),
    "image_roots": {name: str(root) for name, root in IMAGE_ROOTS.items()},
    "legacy": {
        "multitask_cv_dir": str(LEGACY_MULTITASK_CV_DIR),
        "bbox_adapter_dir": str(LEGACY_BBOX_ADAPTER_DIR),
        "multitask_adapter_root": str(LEGACY_MULTITASK_ADAPTER_ROOT),
    },
    "model_revisions": {
        entry["repo_id"]: entry["revision"] for entry in model_registry
    },
}
write_json(NB00_DIR / "stage_a_paths.json", stage_a_paths)

environment_manifest = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "00_environment_and_model_registry.ipynb",
    "protocol_version": "protocol-v1.0",
    "git_commit": git_commit(),
    "host": socket.gethostname(),
    "slurm": {
        key: os.environ.get(key) for key in
        ["SLURM_JOB_ID", "SLURM_JOB_PARTITION", "SLURM_JOB_NODELIST",
         "SLURM_CPUS_ON_NODE", "SLURM_MEM_PER_NODE", "CUDA_VISIBLE_DEVICES"]
    },
    "python": sys.version,
    "platform": platform.platform(),
    "packages": package_versions,
    "torch": {
        "version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
        "bf16_supported": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    },
    "determinism": determinism,
    "gpu_benchmark": gpu_benchmark,
    "huggingface": {
        "identity": HF_IDENTITY.get("name") if HF_IDENTITY else None,
        "auth_error": HF_AUTH_ERROR,
        "offline_mode": is_offline_mode(),
        "hf_home": os.environ.get("HF_HOME"),
        "hf_hub_cache": os.environ.get("HF_HUB_CACHE"),
    },
    "datasets": dataset_records,
    "image_roots": image_root_status,
    "legacy_artifacts": legacy_status,
    "disk_free_gib": round(disk.free / 1024 ** 3, 1) if disk is not None else None,
}
write_json(NB00_DIR / "environment_manifest.json", environment_manifest)
write_json(NB00_DIR / "gpu_benchmark.json", gpu_benchmark)
write_json(NB00_DIR / "model_registry.json", model_registry)

registry_fields = [
    "repo_id", "role", "backend", "dtype", "required", "revision",
    "resolve_status", "load_status", "total_parameters", "total_parameters_billions",
    "load_seconds", "peak_memory_gib", "error", "note",
]
registry_csv = NB00_DIR / "model_registry.csv"
with registry_csv.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=registry_fields)
    writer.writeheader()
    for entry in model_registry:
        row = {field: entry.get(field) for field in registry_fields}
        for field in ["total_parameters", "total_parameters_billions", "load_seconds",
                      "peak_memory_gib", "error"]:
            if row.get(field) is None:
                row[field] = entry["detail"].get(field)
        writer.writerow(row)

print("Wrote:")
for name in ["stage_a_paths.json", "environment_manifest.json", "gpu_benchmark.json",
             "model_registry.json", "model_registry.csv"]:
    print("  ", NB00_DIR / name)

## 9. Gate

A red gate here blocks NB 01-04. Do not comment out an assertion to move on: fix the cause,
or change `required` in `MODEL_REGISTRY_SPEC` deliberately and record why in the
protocol-deviation log.

In [ ]:
failures = []
warnings = []

missing_keys = sorted(set(REQUIRED_DATASET_FILES) - set(dataset_paths))
if missing_keys:
    failures.append(f"Unresolved dataset files: {missing_keys}")

for name, status in image_root_status.items():
    if not status["exists"]:
        failures.append(f"Image root missing: {name} -> {status['path']}")

if not torch.cuda.is_available():
    warnings.append("No CUDA device on this node. Model load checks are unreliable; "
                    "re-run NB 00 on the GPU node that will run Stages B-C.")
elif not torch.cuda.is_bf16_supported():
    failures.append("BF16 is not supported on this GPU. Every LoRA arm assumes BF16.")

for entry in model_registry:
    label = entry["repo_id"]
    if entry["resolve_status"] != "OK":
        message = f"{label}: revision not resolved ({entry['detail'].get('resolve_error')})"
        (failures if entry["required"] else warnings).append(message)
        continue
    if not entry["revision"]:
        (failures if entry["required"] else warnings).append(f"{label}: empty revision SHA")
    if RUN_MODEL_LOAD_CHECK and entry["load_status"] not in {"OK", "DOWNLOADED"}:
        message = f"{label}: load/forward failed ({entry['detail'].get('error')})"
        (failures if entry["required"] else warnings).append(message)

if not legacy_status["bbox_adapter_dir"]["exists"]:
    failures.append(
        f"Anatomy bbox adapter missing at {LEGACY_BBOX_ADAPTER_DIR}. NB 04 cannot run."
    )
if len(legacy_status["multitask_adapter_root"]["folds_with_best_adapter"]) < 5:
    warnings.append(
        "Fewer than five legacy multitask best_adapter directories were found. "
        "Stage B will retrain them on the regenerated folds anyway, so this is "
        "informational unless you intend to reproduce the earlier numbers."
    )

print("WARNINGS")
if warnings:
    for message in warnings:
        print("  -", message)
else:
    print("  none")

print()
print("FAILURES")
if failures:
    for message in failures:
        print("  -", message)
else:
    print("  none")

write_json(NB00_DIR / "gate_nb00.json", {
    "passed": not failures, "failures": failures, "warnings": warnings,
})

assert not failures, f"NB 00 gate failed with {len(failures)} blocking issue(s); see above."
print()
print("NB 00 gate: PASSED")

## Notes carried forward

- Every later notebook opens `stage_a_paths.json` and uses `model_revisions[repo_id]` as its
  `revision=` argument. No notebook downstream of here may pass `revision="main"`.
- `model_registry.csv` is manuscript Table S3. It already contains the parameter counts and
  load times needed for Table 10's operational column.
- The registry holds **four** models. `google/cxr-foundation` was withdrawn per protocol
  D2a as redundant with the MedGemma-1.5 vision tower, which also removes the only
  TensorFlow/Keras dependency from the study. The `download_only` backend is retained in
  `load_and_probe` as generic infrastructure for any future non-transformers checkpoint.
- `m42-health/CXformer-base` is now the sole frozen-encoder agent and carries RQ3 by itself.
  It is listed as `required: False` only because its loader is the least standard of the
  four; if it fails here, resolve it before Stage B rather than proceeding without it.
- If `use_deterministic_algorithms` reports False, the cause is almost always that CUDA was
  initialised before `CUBLAS_WORKSPACE_CONFIG` was exported. Restart the kernel and re-run.